# Manga / Webtoon → Dialogue Text (Colab)

파이프라인: `PDF/이미지 → Koharu RF-DETR → OCR → 언어 판별 → 외국어만 Qwen 번역 → JSONL/TXT`

- 업로드 입력은 `단일 이미지 / 여러 이미지 / PDF / 혼합 입력`을 자동 판별합니다.
- 업로드 직후 이미지 또는 PDF 첫 페이지들의 썸네일을 확인할 수 있습니다.
- PDF 렌더링과 여러 이미지 준비는 CPU 스레드로 병렬 처리합니다.
- 기본값은 일본 만화용 `MangaOCR + RTL`입니다.


## 1. 패키지 설치 + 저장소 가져오기


In [ ]:
!pip -q install \
    "rfdetr==1.7.0" \
    "safetensors>=0.5" \
    "huggingface_hub>=0.27" \
    "manga-ocr>=0.1.11" \
    "paddlepaddle>=3.0" \
    "paddleocr>=3.0" \
    "transformers>=4.51" \
    "accelerate>=1.2" \
    "bitsandbytes>=0.45" \
    "lingua-language-detector>=2.0" \
    "pymupdf>=1.24" \
    pillow numpy tqdm matplotlib

!rm -rf /content/manga2text_tmp
!git clone -q https://github.com/HisameOgasahara/manga2text_tmp.git /content/manga2text_tmp


## 2. 설정

여기만 바꾸면 대부분의 동작을 조절할 수 있습니다.


In [ ]:
import sys
from pathlib import Path

sys.path.append("/content/manga2text_tmp")

from manga2text_pipeline import (
    build_language_detector,
    classify_inputs,
    collect_page_images,
    describe_input_mode,
    load_koharu_detector,
    load_ocr_backend,
    load_translation_model,
    make_preview_images,
    process_pages,
    save_results,
)

WORK_DIR = Path("/content/manga2text")
INPUT_DIR = WORK_DIR / "input"
PAGE_DIR = WORK_DIR / "pages"
OUTPUT_DIR = WORK_DIR / "output"

for directory in [INPUT_DIR, PAGE_DIR, OUTPUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# OCR: "manga" 또는 "paddle"
OCR_BACKEND = "manga"

# PaddleOCR 사용 시: "korean", "japan", "ch", "en" 등
PADDLE_LANG = "korean"
PADDLE_DEVICE = "cpu"

# 일본 만화: "rtl", 한국 웹툰/영문 코믹: "ltr"
READING_DIRECTION = "rtl"
ROW_TOLERANCE = 80

# 효과음까지 OCR하려면 True
INCLUDE_SFX = False
CROP_PADDING = 8

CLASS_THRESHOLDS = {
    0: 0.25,  # text
    1: 0.20,  # onomatopoeia / SFX
    2: 0.50,  # bubble
    3: 0.50,  # panel
}

ENABLE_TRANSLATION = True
TRANSLATION_MODEL = "Qwen/Qwen3-1.7B"
# TRANSLATION_MODEL = "Qwen/Qwen3-4B"
MAX_NEW_TOKENS = 256

PDF_DPI = 200
PAGE_LIMIT = None  # 예: 10이면 각 PDF 앞 10페이지만 테스트

# PDF 페이지 렌더링 / 여러 이미지 준비용 CPU worker 수
# Colab에서는 4 정도부터 시작하는 것을 권장
INPUT_WORKERS = 4

# 업로드 직후 미리보기 개수
PREVIEW_MAX_ITEMS = 8
PDF_PREVIEW_PAGES = 3


## 3. 이미지 / PDF 업로드 + 자동 판별 + 썸네일 확인

한 장만 올려도 되고, 여러 이미지를 한꺼번에 선택해도 되고, PDF를 올려도 됩니다.
PDF와 이미지를 함께 올리는 혼합 입력도 처리합니다.

**주의:** 이 셀을 다시 실행하면 이전 업로드는 지우고 새 입력만 사용합니다.


In [ ]:
import math
import shutil

import matplotlib.pyplot as plt
from google.colab import files

# 이전 입력이 섞이지 않도록 업로드 폴더를 비웁니다.
if INPUT_DIR.exists():
    shutil.rmtree(INPUT_DIR)

INPUT_DIR.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()

for filename, data in uploaded.items():
    destination = INPUT_DIR / filename
    destination.write_bytes(data)

groups = classify_inputs(INPUT_DIR)
input_mode = describe_input_mode(groups)

print(f"입력 형태: {input_mode}")
print(f"이미지: {len(groups['images'])}개")
print(f"PDF: {len(groups['pdfs'])}개")

if groups["unsupported"]:
    print("\n지원하지 않는 파일:")
    for path in groups["unsupported"]:
        print(" -", path.name)

previews = make_preview_images(
    input_dir=INPUT_DIR,
    max_items=PREVIEW_MAX_ITEMS,
    pdf_preview_pages=PDF_PREVIEW_PAGES,
)

if not previews:
    raise RuntimeError("미리볼 수 있는 이미지 또는 PDF가 없습니다.")

column_count = min(4, len(previews))
row_count = math.ceil(len(previews) / column_count)

plt.figure(figsize=(4 * column_count, 4 * row_count))

for index, (label, image) in enumerate(previews, start=1):
    ax = plt.subplot(row_count, column_count, index)
    ax.imshow(image)
    ax.set_title(label, fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()


## 4. 입력 → 페이지 이미지 준비

- 이미지 파일은 처리 폴더로 복사합니다.
- PDF는 페이지별 이미지로 변환합니다.
- 이 단계는 `INPUT_WORKERS`만큼 CPU 스레드를 사용합니다.
- GPU 모델 추론은 한 T4에서 충돌하지 않도록 뒤 단계에서 순차 실행합니다.


In [ ]:
# 이전 페이지 캐시를 지워서 현재 업로드만 처리합니다.
if PAGE_DIR.exists():
    shutil.rmtree(PAGE_DIR)

PAGE_DIR.mkdir(parents=True, exist_ok=True)

page_paths = collect_page_images(
    input_dir=INPUT_DIR,
    page_dir=PAGE_DIR,
    pdf_dpi=PDF_DPI,
    page_limit=PAGE_LIMIT,
    workers=INPUT_WORKERS,
)

print(f"처리할 페이지 수: {len(page_paths)}")

for path in page_paths[:20]:
    print(" -", path)


## 5. Koharu RF-DETR 다운로드 + 로드

공개 모델 `mayocream/koharu-layout-rfdetr-seg-2xl-1152`의 `model.safetensors`와 공식 loader를 Hugging Face에서 자동 다운로드합니다.


In [ ]:
import torch

detector = load_koharu_detector()

print("RF-DETR 로드 완료")
print("CUDA 사용 가능:", torch.cuda.is_available())


## 6. OCR 모델 로드

- `OCR_BACKEND="manga"` → MangaOCR
- `OCR_BACKEND="paddle"` → PaddleOCR


In [ ]:
ocr_model = load_ocr_backend(
    backend=OCR_BACKEND,
    paddle_lang=PADDLE_LANG,
    paddle_device=PADDLE_DEVICE,
)

print("OCR 로드 완료:", OCR_BACKEND)


## 7. 언어 판별기 로드

한글/가나 문자 범위를 먼저 확인하고, 애매한 경우 Lingua로 한국어·일본어·중국어·영어를 판별합니다.


In [ ]:
language_detector, language_to_code = build_language_detector()
print("언어 판별기 로드 완료")


## 8. 소형 번역 LLM 로드

기본은 `Qwen3-1.7B` 4-bit입니다. 한국어로 판별된 텍스트는 번역 단계를 건너뜁니다.


In [ ]:
translation_tokenizer = None
translation_model = None

if ENABLE_TRANSLATION:
    translation_tokenizer, translation_model = load_translation_model(
        model_name=TRANSLATION_MODEL,
    )
    print("번역 모델 로드 완료:", TRANSLATION_MODEL)
else:
    print("번역 비활성화")


## 9. 전체 파이프라인 실행

페이지별로 `RF-DETR 검출 → 읽기 순서 근사 → OCR → 언어 판별 → 외국어만 한국어 번역`을 수행합니다.

GPU 모델 하나를 여러 스레드에서 동시에 호출하지 않도록 이 단계는 기본적으로 순차 처리합니다.


In [ ]:
records = process_pages(
    page_paths=page_paths,
    detector=detector,
    ocr_backend=OCR_BACKEND,
    ocr_model=ocr_model,
    language_detector=language_detector,
    language_to_code=language_to_code,
    class_thresholds=CLASS_THRESHOLDS,
    reading_direction=READING_DIRECTION,
    row_tolerance=ROW_TOLERANCE,
    crop_padding=CROP_PADDING,
    include_sfx=INCLUDE_SFX,
    enable_translation=ENABLE_TRANSLATION,
    translation_tokenizer=translation_tokenizer,
    translation_model=translation_model,
    max_new_tokens=MAX_NEW_TOKENS,
)

print(f"추출된 텍스트 영역: {len(records)}")


## 10. 결과 미리보기


In [ ]:
for record in records[:30]:
    print(
        f"[p.{record['page']:03d} / {record['order']:02d}] "
        f"{record['language']} | "
        f"{record['original']} "
        f"-> {record['korean']}"
    )


## 11. JSONL / TXT 저장 + 다운로드


In [ ]:
jsonl_path, txt_path = save_results(
    records=records,
    output_dir=OUTPUT_DIR,
)

print("저장 완료")
print(" -", jsonl_path)
print(" -", txt_path)

files.download(str(jsonl_path))
files.download(str(txt_path))
